# Celeb-DF ArcFace baseline 재현성·누수 감사

이 노트북은 GitHub Issue #4의 고정 프로토콜을 실행한다.

- `frames/video`: 1, 5, 10
- 등록 영상: 1, 3, 5개
- subject/protocol seed: 5개
- 모든 등록 프로토콜의 query는 공통으로 6번째 영상부터 사용
- validation/test identity와 registration/query video의 교집합을 0으로 검증
- 원본, frame, crop, 개별 score, embedding은 runtime 밖으로 내보내지 않음
- Drive에는 집계 JSON/CSV/PNG와 hash만 포함한 ZIP만 저장

이 결과는 **Celeb-real 동일인 검증 baseline**이며 딥페이크 탐지 성능이 아니다.

In [ ]:
#@title 1. 실행 설정과 권한 확인
REPO_URL = "https://github.com/Chunbae-A/face-image.git" #@param {type:"string"}
BRANCH = "exp/4-celebdf-baseline-audit" #@param {type:"string"}
CODE_SOURCE = "embedded" #@param ["embedded", "github"]
SOURCE_ZIP_PATH = "/content/drive/MyDrive/Celeb-DF-v2.zip" #@param {type:"string"}
# Drive/Drive API가 막혀 로컬 ZIP을 /content에 분할 업로드한 경우만 True.
ASSEMBLE_RUNTIME_UPLOAD_PARTS = False #@param {type:"boolean"}
# 0이면 생략. 분할 전 로컬 ZIP의 실제 바이트를 입력하면 결합 후 검증한다.
EXPECTED_SOURCE_ZIP_BYTES = 0 #@param {type:"integer"}
# DriveFS mount가 반복 실패할 때만 Drive 웹의 파일 ID를 입력한다. Git/결과 bundle에는 기록되지 않는다.
DRIVE_SOURCE_FILE_ID = "" #@param {type:"string"}
DRIVE_RESULT_DIR = "/content/drive/MyDrive/face-image-celebdf-audit" #@param {type:"string"}
PERSIST_SANITIZED_RESULTS_TO_DRIVE = True #@param {type:"boolean"}

# 공식 신청·승인 파일이며 약관상 Hosted Colab/Drive 처리가 허용됨을 직접 확인한 경우만 True.
I_CONFIRM_CELEBDF_CLOUD_PROCESSING_IS_ALLOWED = False #@param {type:"boolean"}
# InsightFace 제공 buffalo_l 가중치는 비상업 연구 전용.
I_ACCEPT_INSIGHTFACE_NONCOMMERCIAL_RESEARCH_LICENSE = False #@param {type:"boolean"}

FRAMES_PER_VIDEO_VALUES = "1,5,10" #@param {type:"string"}
REFERENCE_COUNTS = "1,3,5" #@param {type:"string"}
SEEDS = "20260805,20260806,20260807,20260808,20260809" #@param {type:"string"}
BOOTSTRAP_REPEATS = 500 #@param {type:"integer"}
RUN_SMOKE_BEFORE_FULL = True #@param {type:"boolean"}

import sys
IN_HOSTED_COLAB = "google.colab" in sys.modules
if IN_HOSTED_COLAB and not I_CONFIRM_CELEBDF_CLOUD_PROCESSING_IS_ALLOWED:
    raise PermissionError("Confirm Celeb-DF Hosted Colab/Drive processing permission first.")
if not I_ACCEPT_INSIGHTFACE_NONCOMMERCIAL_RESEARCH_LICENSE:
    raise PermissionError("Accept the InsightFace non-commercial research weight license first.")

FRAME_VALUES = tuple(int(value) for value in FRAMES_PER_VIDEO_VALUES.split(","))
REFERENCE_VALUES = tuple(int(value) for value in REFERENCE_COUNTS.split(","))
SEED_VALUES = tuple(int(value) for value in SEEDS.split(","))
if FRAME_VALUES != (1, 5, 10):
    raise ValueError("Issue #4 protocol requires frames 1,5,10.")
if REFERENCE_VALUES != (1, 3, 5):
    raise ValueError("Issue #4 protocol requires references 1,3,5.")
if len(SEED_VALUES) != 5 or len(set(SEED_VALUES)) != 5:
    raise ValueError("Issue #4 protocol requires five unique seeds.")
print({
    "hosted_colab": IN_HOSTED_COLAB,
    "frames": FRAME_VALUES,
    "references": REFERENCE_VALUES,
    "seeds": SEED_VALUES,
    "maximum_frame_inferences": 590 * sum(FRAME_VALUES),
})

## 실행 환경

Colab에서 GPU runtime을 선택한다. 설치 셀은 Colab CUDA 사용자 라이브러리와 호환되는 `onnxruntime-gpu==1.23.2`를 고정한다. 설치 후 runtime을 재시작했다면 2번 설치 셀을 건너뛰고 1번과 3번 이후 셀을 다시 실행한다.

In [ ]:
#@title 2. 라이브러리 설치
%pip uninstall -y -q onnxruntime onnxruntime-gpu
%pip install -q --no-cache-dir "insightface==1.0.1" "onnxruntime-gpu==1.23.2" opencv-python-headless pandas matplotlib seaborn tqdm

In [ ]:
#@title 3. 실행 코드 준비 — 기본값은 PR 코드가 내장된 embedded 모드
from pathlib import Path
import base64
import os
import subprocess

EMBEDDED_FILES_B64 = {'scripts/celebdf_faceguard.py': '...', 'scripts/run_celebdf_arcface.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJFeHRyYWN0IHZpZGVvLWxldmVsIEFyY0ZhY2UgZW1iZWRkaW5ncyBmcm9tIENlbGViLURGLXYyIENlbGViLXJlYWwgdmlkZW9zLgoKVGhpcyBydW5uZXIgaXMgaW50ZW5kZWQgZm9yIEdvb2dsZSBDb2xhYiBvciBhbm90aGVyIGVudmlyb25tZW50IHdpdGggT3BlbkNWLApJbnNpZ2h0RmFjZSwgYW5kIE9OTlggUnVudGltZSBpbnN0YWxsZWQuICBJdCBuZXZlciBzYXZlcyBmYWNlIGNyb3BzIG9yIHNhbXBsZWQKZnJhbWVzLiAgRXZlcnkgc3VjY2Vzc2Z1bCB2aWRlbyBwcm9kdWNlcyBvbmUgbm9ybWFsaXplZCA1MTItRCBlbWJlZGRpbmcsIGFuZAp0aGUgTlBaIGNoZWNrcG9pbnQgaXMgYXRvbWljYWxseSByZXBsYWNlZCBhdCBhIGNvbmZpZ3VyYWJsZSBpbnRlcnZhbC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGNzdgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHRpbWUKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgYXNkaWN0CmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLCB0aW1lem9uZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueSwgU2VxdWVuY2UKCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSBjZWxlYmRmX2ZhY2VndWFyZCBpbXBvcnQgKAogICAgQXJjaGl2ZVZpZGVvLAogICAgVmlkZW9FbWJlZGRpbmcsCiAgICBsMl9ub3JtYWxpemUsCiAgICBsb2FkX3ZpZGVvX2VtYmVkZGluZ3MsCiAgICByZWFkX21hbmlmZXN0LAogICAgc2F2ZV92aWRlb19lbWJlZGRpbmdzLAogICAgc2VsZWN0X3Ntb2tlX3Jvd3MsCikKCgpkZWYgc2FtcGxlX2ZyYW1lX2luZGljZXMoZnJhbWVfY291bnQ6IGludCwgcmVxdWVzdGVkOiBpbnQpIC0+IGxpc3RbaW50XToKICAgICIiIlJldHVybiB1bmlxdWUsIGV2ZW5seSBzcGFjZWQgZnJhbWUgaW5kaWNlcyB3aGlsZSBhdm9pZGluZyBoYXJkIGN1dHMgYXQgZW5kcy4iIiIKICAgIGlmIGZyYW1lX2NvdW50IDw9IDAgb3IgcmVxdWVzdGVkIDw9IDA6CiAgICAgICAgcmV0dXJuIFtdCiAgICBpZiBmcmFtZV9jb3VudCA8PSByZXF1ZXN0ZWQ6CiAgICAgICAgcmV0dXJuIGxpc3QocmFuZ2UoZnJhbWVfY291bnQpKQogICAgZmlyc3QgPSBtaW4oZnJhbWVfY291bnQgLSAxLCBtYXgoMCwgaW50KHJvdW5kKGZyYW1lX2NvdW50ICogMC4wOCkpKSkKICAgIGxhc3QgPSBtYXgoZmlyc3QsIG1pbihmcmFtZV9jb3VudCAtIDEsIGludChyb3VuZChmcmFtZV9jb3VudCAqIDAuOTIpKSAtIDEpKQogICAgaW5kaWNlcyA9IG5wLmxpbnNwYWNlKGZpcnN0LCBsYXN0LCBudW09cmVxdWVzdGVkLCBkdHlwZT1pbnQpCiAgICByZXR1cm4gc29ydGVkKHNldChpbnQoaW5kZXgpIGZvciBpbmRleCBpbiBpbmRpY2VzKSkKCgpkZWYgX2ZhY2VfYXJlYV9yYXRpbyhmYWNlOiBBbnksIGZyYW1lX3NoYXBlOiBTZXF1ZW5jZVtpbnRdKSAtPiBmbG9hdDoKICAgIGhlaWdodCwgd2lkdGggPSBpbnQoZnJhbWVfc2hhcGVbMF0pLCBpbnQoZnJhbWVfc2hhcGVbMV0pCiAgICBpZiBoZWlnaHQgPD0gMCBvciB3aWR0aCA8PSAwOgogICAgICAgIHJldHVybiAwLjAKICAgIGxlZnQsIHRvcCwgcmlnaHQsIGJvdHRvbSA9IFtmbG9hdCh2YWx1ZSkgZm9yIHZhbHVlIGluIGZhY2UuYmJveF0KICAgIGFyZWEgPSBtYXgoMC4wLCByaWdodCAtIGxlZnQpICogbWF4KDAuMCwgYm90dG9tIC0gdG9wKQogICAgcmV0dXJuIGFyZWEgLyBmbG9hdChoZWlnaHQgKiB3aWR0aCkKCgpkZWYgc2VsZWN0X3ByaW1hcnlfZmFjZSgKICAgIGZhY2VzOiBTZXF1ZW5jZVtBbnldLAogICAgZnJhbWVfc2hhcGU6IFNlcXVlbmNlW2ludF0sCiAgICBydW5uaW5nX3RlbXBsYXRlOiBucC5uZGFycmF5IHwgTm9uZSwKKSAtPiBBbnkgfCBOb25lOgogICAgIiIiQ2hvb3NlIHRoZSBsYXJnZXN0IGZpcnN0IGZhY2UsIHRoZW4gdHJhY2sgYnkgZW1iZWRkaW5nIHNpbWlsYXJpdHkuIiIiCiAgICBjYW5kaWRhdGVzID0gWwogICAgICAgIGZhY2UgZm9yIGZhY2UgaW4gZmFjZXMgaWYgZ2V0YXR0cihmYWNlLCAibm9ybWVkX2VtYmVkZGluZyIsIE5vbmUpIGlzIG5vdCBOb25lCiAgICBdCiAgICBpZiBub3QgY2FuZGlkYXRlczoKICAgICAgICByZXR1cm4gTm9uZQogICAgaWYgcnVubmluZ190ZW1wbGF0ZSBpcyBOb25lOgogICAgICAgIHJldHVybiBtYXgoY2FuZGlkYXRlcywga2V5PWxhbWJkYSBmYWNlOiBfZmFjZV9hcmVhX3JhdGlvKGZhY2UsIGZyYW1lX3NoYXBlKSkKICAgIHRlbXBsYXRlID0gbDJfbm9ybWFsaXplKHJ1bm5pbmdfdGVtcGxhdGUpCiAgICByZXR1cm4gbWF4KAogICAgICAgIGNhbmRpZGF0ZXMsCiAgICAgICAga2V5PWxhbWJkYSBmYWNlOiBmbG9hdChsMl9ub3JtYWxpemUoZmFjZS5ub3JtZWRfZW1iZWRkaW5nKSBAIHRlbXBsYXRlKSwKICAgICkKCgpkZWYgZW1iZWRfdmlkZW8oCiAgICB2aWRlb19wYXRoOiBQYXRoLAogICAgcm93OiBBcmNoaXZlVmlkZW8sCiAgICBmYWNlX2FwcDogQW55LAogICAgKiwKICAgIGZyYW1lc19wZXJfdmlkZW86IGludCwKICAgIG1pbmltdW1fdmFsaWRfZnJhbWVzOiBpbnQsCikgLT4gdHVwbGVbVmlkZW9FbWJlZGRpbmcgfCBOb25lLCBkaWN0W3N0ciwgb2JqZWN0XSB8IE5vbmVdOgogICAgaW1wb3J0IGN2MiAgIyB0eXBlOiBpZ25vcmUKCiAgICBjYXB0dXJlID0gY3YyLlZpZGVvQ2FwdHVyZShzdHIodmlkZW9fcGF0aCkpCiAgICBpZiBub3QgY2FwdHVyZS5pc09wZW5lZCgpOgogICAgICAgIHJldHVybiBOb25lLCB7InZpZGVvX2lkIjogcm93LnZpZGVvX2lkLCAicmVhc29uIjogInZpZGVvX29wZW5fZmFpbGVkIn0KICAgIHRyeToKICAgICAgICBmcmFtZV9jb3VudCA9IGludChjYXB0dXJlLmdldChjdjIuQ0FQX1BST1BfRlJBTUVfQ09VTlQpKQogICAgICAgIGluZGljZXMgPSBzYW1wbGVfZnJhbWVfaW5kaWNlcyhmcmFtZV9jb3VudCwgZnJhbWVzX3Blcl92aWRlbykKICAgICAgICBpZiBub3QgaW5kaWNlczoKICAgICAgICAgICAgcmV0dXJuIE5vbmUsIHsidmlkZW9faWQiOiByb3cudmlkZW9faWQsICJyZWFzb24iOiAiaW52YWxpZF9mcmFtZV9jb3VudCJ9CgogICAgICAgIGVtYmVkZGluZ3M6IGxpc3RbbnAubmRhcnJheV0gPSBbXQogICAgICAgIGRldGVjdGlvbl9zY29yZXM6IGxpc3RbZmxvYXRdID0gW10KICAgICAgICBmYWNlX2FyZWFfcmF0aW9zOiBsaXN0W2Zsb2F0XSA9IFtdCiAgICAgICAgZGVjb2RlX3NlY29uZHMgPSAwLjAKICAgICAgICBpbmZlcmVuY2Vfc2Vjb25kcyA9IDAuMAogICAgICAgIGZvciBmcmFtZV9pbmRleCBpbiBpbmRpY2VzOgogICAgICAgICAgICBkZWNvZGVfc3RhcnQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgICAgIGNhcHR1cmUuc2V0KGN2Mi5DQVBfUFJPUF9QT1NfRlJBTUVTLCBmcmFtZV9pbmRleCkKICAgICAgICAgICAgb2ssIGZyYW1lID0gY2FwdHVyZS5yZWFkKCkKICAgICAgICAgICAgZGVjb2RlX3NlY29uZHMgKz0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIGRlY29kZV9zdGFydAogICAgICAgICAgICBpZiBub3Qgb2sgb3IgZnJhbWUgaXMgTm9uZToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgICAgICBpbmZlcmVuY2Vfc3RhcnQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgICAgIGZhY2VzID0gZmFjZV9hcHAuZ2V0KGZyYW1lKQogICAgICAgICAgICBpbmZlcmVuY2Vfc2Vjb25kcyArPSB0aW1lLnBlcmZfY291bnRlcigpIC0gaW5mZXJlbmNlX3N0YXJ0CiAgICAgICAgICAgIHJ1bm5pbmdfdGVtcGxhdGUgPSAoCiAgICAgICAgICAgICAgICBsMl9ub3JtYWxpemUobnAubWVhbihucC5zdGFjayhlbWJlZGRpbmdzKSwgYXhpcz0wKSkKICAgICAgICAgICAgICAgIGlmIGVtYmVkZGluZ3MKICAgICAgICAgICAgICAgIGVsc2UgTm9uZQogICAgICAgICAgICApCiAgICAgICAgICAgIHNlbGVjdGVkID0gc2VsZWN0X3ByaW1hcnlfZmFjZShmYWNlcywgZnJhbWUuc2hhcGUsIHJ1bm5pbmdfdGVtcGxhdGUpCiAgICAgICAgICAgIGlmIHNlbGVjdGVkIGlzIE5vbmU6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBlbWJlZGRpbmdzLmFwcGVuZChsMl9ub3JtYWxpemUoc2VsZWN0ZWQubm9ybWVkX2VtYmVkZGluZykpCiAgICAgICAgICAgIGRldGVjdGlvbl9zY29yZXMuYXBwZW5kKGZsb2F0KGdldGF0dHIoc2VsZWN0ZWQsICJkZXRfc2NvcmUiLCBucC5uYW4pKSkKICAgICAgICAgICAgZmFjZV9hcmVhX3JhdGlvcy5hcHBlbmQoX2ZhY2VfYXJlYV9yYXRpbyhzZWxlY3RlZCwgZnJhbWUuc2hhcGUpKQoKICAgICAgICBpZiBsZW4oZW1iZWRkaW5ncykgPCBtaW5pbXVtX3ZhbGlkX2ZyYW1lczoKICAgICAgICAgICAgcmV0dXJuIE5vbmUsIHsKICAgICAgICAgICAgICAgICJ2aWRlb19pZCI6IHJvdy52aWRlb19pZCwKICAgICAgICAgICAgICAgICJyZWFzb24iOiAiaW5zdWZmaWNpZW50X3ZhbGlkX2ZhY2VzIiwKICAgICAgICAgICAgICAgICJzYW1wbGVkX2ZyYW1lcyI6IGxlbihpbmRpY2VzKSwKICAgICAgICAgICAgICAgICJ2YWxpZF9mcmFtZXMiOiBsZW4oZW1iZWRkaW5ncyksCiAgICAgICAgICAgIH0KICAgICAgICByZXR1cm4gKAogICAgICAgICAgICBWaWRlb0VtYmVkZGluZygKICAgICAgICAgICAgICAgIHN1YmplY3RfaWQ9cm93LnN1YmplY3RfaWQsCiAgICAgICAgICAgICAgICB2aWRlb19pZD1yb3cudmlkZW9faWQsCiAgICAgICAgICAgICAgICByZWxhdGl2ZV9wYXRoPXJvdy5yZWxhdGl2ZV9wYXRoLAogICAgICAgICAgICAgICAgZW1iZWRkaW5nPWwyX25vcm1hbGl6ZShucC5tZWFuKG5wLnN0YWNrKGVtYmVkZGluZ3MpLCBheGlzPTApKSwKICAgICAgICAgICAgICAgIHNhbXBsZWRfZnJhbWVzPWxlbihpbmRpY2VzKSwKICAgICAgICAgICAgICAgIHZhbGlkX2ZyYW1lcz1sZW4oZW1iZWRkaW5ncyksCiAgICAgICAgICAgICAgICBtZWFuX2RldGVjdGlvbl9zY29yZT1mbG9hdChucC5uYW5tZWFuKGRldGVjdGlvbl9zY29yZXMpKSwKICAgICAgICAgICAgICAgIG1lYW5fZmFjZV9hcmVhX3JhdGlvPWZsb2F0KG5wLm1lYW4oZmFjZV9hcmVhX3JhdGlvcykpLAogICAgICAgICAgICAgICAgZGVjb2RlX3NlY29uZHM9ZGVjb2RlX3NlY29uZHMsCiAgICAgICAgICAgICAgICBpbmZlcmVuY2Vfc2Vjb25kcz1pbmZlcmVuY2Vfc2Vjb25kcywKICAgICAgICAgICAgKSwKICAgICAgICAgICAgTm9uZSwKICAgICAgICApCiAgICBmaW5hbGx5OgogICAgICAgIGNhcHR1cmUucmVsZWFzZSgpCgoKZGVmIF9zaGEyNTYocGF0aDogUGF0aCkgLT4gc3RyOgogICAgZGlnZXN0ID0gaGFzaGxpYi5zaGEyNTYoKQogICAgd2l0aCBwYXRoLm9wZW4oInJiIikgYXMgaGFuZGxlOgogICAgICAgIGZvciBjaHVuayBpbiBpdGVyKGxhbWJkYTogaGFuZGxlLnJlYWQoMTAyNCAqIDEwMjQpLCBiIiIpOgogICAgICAgICAgICBkaWdlc3QudXBkYXRlKGNodW5rKQogICAgcmV0dXJuIGRpZ2VzdC5oZXhkaWdlc3QoKQoKCmRlZiBfZ2l0X2NvbW1pdCgpIC0+IHN0ciB8IE5vbmU6CiAgICB0cnk6CiAgICAgICAgcmV0dXJuIHN1YnByb2Nlc3MuY2hlY2tfb3V0cHV0KAogICAgICAgICAgICBbImdpdCIsICJyZXYtcGFyc2UiLCAiSEVBRCJdLAogICAgICAgICAgICB0ZXh0PVRydWUsCiAgICAgICAgICAgIHN0ZGVycj1zdWJwcm9jZXNzLkRFVk5VTEwsCiAgICAgICAgKS5zdHJpcCgpCiAgICBleGNlcHQgKEZpbGVOb3RGb3VuZEVycm9yLCBzdWJwcm9jZXNzLlN1YnByb2Nlc3NFcnJvcik6CiAgICAgICAgcmV0dXJuIE5vbmUKCgpkZWYgX3dyaXRlX3JlamVjdHMocm93czogU2VxdWVuY2VbZGljdFtzdHIsIG9iamVjdF1dLCBwYXRoOiBQYXRoKSAtPiBOb25lOgogICAgaWYgbm90IHJvd3M6CiAgICAgICAgcmV0dXJuCiAgICBmaWVsZHMgPSBzb3J0ZWQoe2tleSBmb3Igcm93IGluIHJvd3MgZm9yIGtleSBpbiByb3d9KQogICAgdGVtcG9yYXJ5ID0gcGF0aC53aXRoX3N1ZmZpeChwYXRoLnN1ZmZpeCArICIudG1wIikKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggdGVtcG9yYXJ5Lm9wZW4oInciLCBuZXdsaW5lPSIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBoYW5kbGU6CiAgICAgICAgd3JpdGVyID0gY3N2LkRpY3RXcml0ZXIoaGFuZGxlLCBmaWVsZG5hbWVzPWZpZWxkcykKICAgICAgICB3cml0ZXIud3JpdGVoZWFkZXIoKQogICAgICAgIHdyaXRlci53cml0ZXJvd3Mocm93cykKICAgIG9zLnJlcGxhY2UodGVtcG9yYXJ5LCBwYXRoKQoKCmRlZiBfbW9kZWxfaGFzaGVzKG1vZGVsX3Jvb3Q6IFBhdGgsIG1vZGVsX25hbWU6IHN0cikgLT4gZGljdFtzdHIsIHN0cl06CiAgICBtb2RlbF9kaXIgPSBtb2RlbF9yb290LmV4cGFuZHVzZXIoKSAvICJtb2RlbHMiIC8gbW9kZWxfbmFtZQogICAgaWYgbm90IG1vZGVsX2Rpci5leGlzdHMoKToKICAgICAgICByZXR1cm4ge30KICAgIHJldHVybiB7CiAgICAgICAgc3RyKHBhdGgucmVsYXRpdmVfdG8obW9kZWxfZGlyKSk6IF9zaGEyNTYocGF0aCkKICAgICAgICBmb3IgcGF0aCBpbiBzb3J0ZWQobW9kZWxfZGlyLnJnbG9iKCIqLm9ubngiKSkKICAgIH0KCgpkZWYgaW5pdGlhbGl6ZV9mYWNlX2FwcChtb2RlbF9uYW1lOiBzdHIsIG1vZGVsX3Jvb3Q6IFBhdGgsIGRldF9zaXplOiBpbnQpIC0+IHR1cGxlW0FueSwgZGljdFtzdHIsIG9iamVjdF1dOgogICAgaW1wb3J0IGluc2lnaHRmYWNlICAjIHR5cGU6IGlnbm9yZQogICAgaW1wb3J0IG9ubnhydW50aW1lIGFzIG9ydCAgIyB0eXBlOiBpZ25vcmUKICAgIGZyb20gaW5zaWdodGZhY2UuYXBwIGltcG9ydCBGYWNlQW5hbHlzaXMgICMgdHlwZTogaWdub3JlCgogICAgYXZhaWxhYmxlID0gb3J0LmdldF9hdmFpbGFibGVfcHJvdmlkZXJzKCkKICAgIHByb3ZpZGVycyA9IFsKICAgICAgICBwcm92aWRlcgogICAgICAgIGZvciBwcm92aWRlciBpbiAoIkNVREFFeGVjdXRpb25Qcm92aWRlciIsICJDUFVFeGVjdXRpb25Qcm92aWRlciIpCiAgICAgICAgaWYgcHJvdmlkZXIgaW4gYXZhaWxhYmxlCiAgICBdCiAgICBpZiBub3QgcHJvdmlkZXJzOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIm5vIHN1cHBvcnRlZCBPTk5YIFJ1bnRpbWUgcHJvdmlkZXIgZm91bmQ6IHthdmFpbGFibGV9IikKICAgIGFwcCA9IEZhY2VBbmFseXNpcygKICAgICAgICBuYW1lPW1vZGVsX25hbWUsCiAgICAgICAgcm9vdD1zdHIobW9kZWxfcm9vdC5leHBhbmR1c2VyKCkpLAogICAgICAgIGFsbG93ZWRfbW9kdWxlcz1bImRldGVjdGlvbiIsICJyZWNvZ25pdGlvbiJdLAogICAgICAgIHByb3ZpZGVycz1wcm92aWRlcnMsCiAgICApCiAgICBjdWRhID0gIkNVREFFeGVjdXRpb25Qcm92aWRlciIgaW4gcHJvdmlkZXJzCiAgICBhcHAucHJlcGFyZSgKICAgICAgICBjdHhfaWQ9MCBpZiBjdWRhIGVsc2UgLTEsCiAgICAgICAgZGV0X3NpemU9KGRldF9zaXplLCBkZXRfc2l6ZSksCiAgICApCiAgICBpbnZlbnRvcnkgPSB7CiAgICAgICAgImluc2lnaHRmYWNlX3ZlcnNpb24iOiBnZXRhdHRyKGluc2lnaHRmYWNlLCAiX192ZXJzaW9uX18iLCAidW5rbm93biIpLAogICAgICAgICJvbm54cnVudGltZV92ZXJzaW9uIjogb3J0Ll9fdmVyc2lvbl9fLAogICAgICAgICJvbm54cnVudGltZV9hdmFpbGFibGVfcHJvdmlkZXJzIjogYXZhaWxhYmxlLAogICAgICAgICJvbm54cnVudGltZV9zZWxlY3RlZF9wcm92aWRlcnMiOiBwcm92aWRlcnMsCiAgICAgICAgImRldmljZSI6ICJjdWRhIiBpZiBjdWRhIGVsc2UgImNwdSIsCiAgICAgICAgIm1vZGVsX25hbWUiOiBtb2RlbF9uYW1lLAogICAgICAgICJtb2RlbF9yb290Ijogc3RyKG1vZGVsX3Jvb3QuZXhwYW5kdXNlcigpKSwKICAgICAgICAibW9kZWxfaGFzaGVzIjogX21vZGVsX2hhc2hlcyhtb2RlbF9yb290LCBtb2RlbF9uYW1lKSwKICAgIH0KICAgIHJldHVybiBhcHAsIGludmVudG9yeQoKCmRlZiBydW5fcGlwZWxpbmUoYXJnczogYXJncGFyc2UuTmFtZXNwYWNlKSAtPiBkaWN0W3N0ciwgb2JqZWN0XToKICAgIGlmIG5vdCBhcmdzLmFjY2VwdF9ub25jb21tZXJjaWFsX21vZGVsX2xpY2Vuc2U6CiAgICAgICAgcmFpc2UgUGVybWlzc2lvbkVycm9yKAogICAgICAgICAgICAiSW5zaWdodEZhY2UtcHJvdmlkZWQgcHJldHJhaW5lZCBtb2RlbHMgYXJlIG5vbi1jb21tZXJjaWFsIHJlc2VhcmNoIG9ubHk7ICIKICAgICAgICAgICAgInBhc3MgLS1hY2NlcHQtbm9uY29tbWVyY2lhbC1tb2RlbC1saWNlbnNlIGFmdGVyIHJldmlld2luZyB0aGUgbGljZW5zZS4iCiAgICAgICAgKQogICAgbWFuaWZlc3Rfcm93cyA9IHJlYWRfbWFuaWZlc3QoYXJncy5tYW5pZmVzdCkKICAgIHNlbGVjdGVkX3Jvd3MgPSBtYW5pZmVzdF9yb3dzCiAgICBpZiBhcmdzLm1vZGUgPT0gInNtb2tlIjoKICAgICAgICBzZWxlY3RlZF9yb3dzID0gc2VsZWN0X3Ntb2tlX3Jvd3MoCiAgICAgICAgICAgIG1hbmlmZXN0X3Jvd3MsCiAgICAgICAgICAgIHN1YmplY3RzPWFyZ3Muc21va2Vfc3ViamVjdHMsCiAgICAgICAgICAgIHZpZGVvc19wZXJfc3ViamVjdD1hcmdzLnNtb2tlX3ZpZGVvc19wZXJfc3ViamVjdCwKICAgICAgICApCgogICAgZXhpc3Rpbmc6IGxpc3RbVmlkZW9FbWJlZGRpbmddID0gW10KICAgIGlmIGFyZ3Mub3V0cHV0LmV4aXN0cygpOgogICAgICAgIGV4aXN0aW5nID0gbG9hZF92aWRlb19lbWJlZGRpbmdzKGFyZ3Mub3V0cHV0KQogICAgY29tcGxldGVkID0ge3JlY29yZC52aWRlb19pZCBmb3IgcmVjb3JkIGluIGV4aXN0aW5nfQogICAgcmVjb3JkcyA9IGxpc3QoZXhpc3RpbmcpCiAgICByZWplY3RzOiBsaXN0W2RpY3Rbc3RyLCBvYmplY3RdXSA9IFtdCgogICAgZmFjZV9hcHAsIHJ1bnRpbWVfaW52ZW50b3J5ID0gaW5pdGlhbGl6ZV9mYWNlX2FwcCgKICAgICAgICBhcmdzLm1vZGVsX25hbWUsCiAgICAgICAgYXJncy5tb2RlbF9yb290LAogICAgICAgIGFyZ3MuZGV0X3NpemUsCiAgICApCiAgICBzdGFydGVkID0gZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykKICAgIHByb2Nlc3NlZF9zaW5jZV9jaGVja3BvaW50ID0gMAogICAgYXR0ZW1wdGVkID0gMAogICAgZm9yIGluZGV4LCByb3cgaW4gZW51bWVyYXRlKHNlbGVjdGVkX3Jvd3MsIHN0YXJ0PTEpOgogICAgICAgIGlmIHJvdy52aWRlb19pZCBpbiBjb21wbGV0ZWQ6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgYXR0ZW1wdGVkICs9IDEKICAgICAgICB2aWRlb19wYXRoID0gYXJncy52aWRlb19yb290IC8gUGF0aChyb3cucmVsYXRpdmVfcGF0aCkKICAgICAgICBpZiBub3QgdmlkZW9fcGF0aC5leGlzdHMoKToKICAgICAgICAgICAgcmVqZWN0cy5hcHBlbmQoeyJ2aWRlb19pZCI6IHJvdy52aWRlb19pZCwgInJlYXNvbiI6ICJ2aWRlb19taXNzaW5nIn0pCiAgICAgICAgICAgIGlmIGFyZ3MuZmFpbF9mYXN0OgogICAgICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IodmlkZW9fcGF0aCkKICAgICAgICAgICAgY29udGludWUKICAgICAgICB0cnk6CiAgICAgICAgICAgIHJlY29yZCwgcmVqZWN0ID0gZW1iZWRfdmlkZW8oCiAgICAgICAgICAgICAgICB2aWRlb19wYXRoLAogICAgICAgICAgICAgICAgcm93LAogICAgICAgICAgICAgICAgZmFjZV9hcHAsCiAgICAgICAgICAgICAgICBmcmFtZXNfcGVyX3ZpZGVvPWFyZ3MuZnJhbWVzX3Blcl92aWRlbywKICAgICAgICAgICAgICAgIG1pbmltdW1fdmFsaWRfZnJhbWVzPWFyZ3MubWluaW11bV92YWxpZF9mcmFtZXMsCiAgICAgICAgICAgICkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICAgICAgaWYgYXJncy5mYWlsX2Zhc3Q6CiAgICAgICAgICAgICAgICByYWlzZQogICAgICAgICAgICByZWNvcmQgPSBOb25lCiAgICAgICAgICAgIHJlamVjdCA9IHsKICAgICAgICAgICAgICAgICJ2aWRlb19pZCI6IHJvdy52aWRlb19pZCwKICAgICAgICAgICAgICAgICJyZWFzb24iOiAidW5leHBlY3RlZF9lcnJvciIsCiAgICAgICAgICAgICAgICAiZXJyb3JfdHlwZSI6IHR5cGUoZXhjKS5fX25hbWVfXywKICAgICAgICAgICAgICAgICJtZXNzYWdlIjogc3RyKGV4YylbOjMwMF0sCiAgICAgICAgICAgIH0KICAgICAgICBpZiByZWNvcmQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHJlY29yZHMuYXBwZW5kKHJlY29yZCkKICAgICAgICAgICAgY29tcGxldGVkLmFkZChyZWNvcmQudmlkZW9faWQpCiAgICAgICAgICAgIHByb2Nlc3NlZF9zaW5jZV9jaGVja3BvaW50ICs9IDEKICAgICAgICBpZiByZWplY3QgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHJlamVjdHMuYXBwZW5kKHJlamVjdCkKCiAgICAgICAgaWYgcHJvY2Vzc2VkX3NpbmNlX2NoZWNrcG9pbnQgPj0gYXJncy5jaGVja3BvaW50X2V2ZXJ5OgogICAgICAgICAgICBzYXZlX3ZpZGVvX2VtYmVkZGluZ3MocmVjb3JkcywgYXJncy5vdXRwdXQpCiAgICAgICAgICAgIF93cml0ZV9yZWplY3RzKHJlamVjdHMsIGFyZ3MucmVqZWN0cykKICAgICAgICAgICAgcHJvY2Vzc2VkX3NpbmNlX2NoZWNrcG9pbnQgPSAwCiAgICAgICAgaWYgaW5kZXggPT0gMSBvciBpbmRleCAlIGFyZ3MucHJvZ3Jlc3NfZXZlcnkgPT0gMCBvciBpbmRleCA9PSBsZW4oc2VsZWN0ZWRfcm93cyk6CiAgICAgICAgICAgIHByaW50KAogICAgICAgICAgICAgICAganNvbi5kdW1wcygKICAgICAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgICAgICJzZWxlY3RlZCI6IGxlbihzZWxlY3RlZF9yb3dzKSwKICAgICAgICAgICAgICAgICAgICAgICAgInZpc2l0ZWQiOiBpbmRleCwKICAgICAgICAgICAgICAgICAgICAgICAgInN1Y2Nlc3NmdWxfdG90YWwiOiBsZW4ocmVjb3JkcyksCiAgICAgICAgICAgICAgICAgICAgICAgICJyZWplY3RlZF90aGlzX3J1biI6IGxlbihyZWplY3RzKSwKICAgICAgICAgICAgICAgICAgICB9LAogICAgICAgICAgICAgICAgICAgIGVuc3VyZV9hc2NpaT1GYWxzZSwKICAgICAgICAgICAgICAgICksCiAgICAgICAgICAgICAgICBmbHVzaD1UcnVlLAogICAgICAgICAgICApCgogICAgaWYgbm90IHJlY29yZHM6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJubyB2aWRlbyBlbWJlZGRpbmdzIHdlcmUgcHJvZHVjZWQiKQogICAgc2F2ZV92aWRlb19lbWJlZGRpbmdzKHJlY29yZHMsIGFyZ3Mub3V0cHV0KQogICAgX3dyaXRlX3JlamVjdHMocmVqZWN0cywgYXJncy5yZWplY3RzKQogICAgZW5kZWQgPSBkYXRldGltZS5ub3codGltZXpvbmUudXRjKQogICAgcmVwb3J0ID0gewogICAgICAgICJzdGF0dXMiOiAiY29tcGxldGVkIiwKICAgICAgICAibW9kZSI6IGFyZ3MubW9kZSwKICAgICAgICAic2VsZWN0ZWRfdmlkZW9fY291bnQiOiBsZW4oc2VsZWN0ZWRfcm93cyksCiAgICAgICAgImF0dGVtcHRlZF90aGlzX3J1biI6IGF0dGVtcHRlZCwKICAgICAgICAic3VjY2Vzc2Z1bF92aWRlb19jb3VudF90b3RhbCI6IGxlbihyZWNvcmRzKSwKICAgICAgICAicmVqZWN0ZWRfdGhpc19ydW4iOiBsZW4ocmVqZWN0cyksCiAgICAgICAgImZyYW1lc19wZXJfdmlkZW8iOiBhcmdzLmZyYW1lc19wZXJfdmlkZW8sCiAgICAgICAgIm1pbmltdW1fdmFsaWRfZnJhbWVzIjogYXJncy5taW5pbXVtX3ZhbGlkX2ZyYW1lcywKICAgICAgICAic3RhcnRlZF91dGMiOiBzdGFydGVkLmlzb2Zvcm1hdCgpLAogICAgICAgICJlbmRlZF91dGMiOiBlbmRlZC5pc29mb3JtYXQoKSwKICAgICAgICAiZWxhcHNlZF9zZWNvbmRzIjogKGVuZGVkIC0gc3RhcnRlZCkudG90YWxfc2Vjb25kcygpLAogICAgICAgICJtYW5pZmVzdCI6IHN0cihhcmdzLm1hbmlmZXN0KSwKICAgICAgICAibWFuaWZlc3Rfc2hhMjU2IjogX3NoYTI1NihhcmdzLm1hbmlmZXN0KSwKICAgICAgICAidmlkZW9fcm9vdCI6IHN0cihhcmdzLnZpZGVvX3Jvb3QpLAogICAgICAgICJvdXRwdXQiOiBzdHIoYXJncy5vdXRwdXQpLAogICAgICAgICJyZWplY3RzIjogc3RyKGFyZ3MucmVqZWN0cyksCiAgICAgICAgImdpdF9jb21taXQiOiBfZ2l0X2NvbW1pdCgpLAogICAgICAgICJtb2RlbF9saWNlbnNlX3Njb3BlIjogIkluc2lnaHRGYWNlLXByb3ZpZGVkIHdlaWdodHM6IG5vbi1jb21tZXJjaWFsIHJlc2VhcmNoIG9ubHkiLAogICAgICAgICoqcnVudGltZV9pbnZlbnRvcnksCiAgICB9CiAgICBhcmdzLnJ1bl9yZXBvcnQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGFyZ3MucnVuX3JlcG9ydC53cml0ZV90ZXh0KAogICAgICAgIGpzb24uZHVtcHMocmVwb3J0LCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0yKSwKICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgKQogICAgcmV0dXJuIHJlcG9ydAoKCmRlZiBidWlsZF9wYXJzZXIoKSAtPiBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcjoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1hbmlmZXN0IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS12aWRlby1yb290IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJlamVjdHMiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJ1bi1yZXBvcnQiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1vZGUiLCBjaG9pY2VzPSgic21va2UiLCAiZnVsbCIpLCBkZWZhdWx0PSJmdWxsIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc21va2Utc3ViamVjdHMiLCB0eXBlPWludCwgZGVmYXVsdD0yKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zbW9rZS12aWRlb3MtcGVyLXN1YmplY3QiLCB0eXBlPWludCwgZGVmYXVsdD0xKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1mcmFtZXMtcGVyLXZpZGVvIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTApCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1pbmltdW0tdmFsaWQtZnJhbWVzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tY2hlY2twb2ludC1ldmVyeSIsIHR5cGU9aW50LCBkZWZhdWx0PTI1KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1wcm9ncmVzcy1ldmVyeSIsIHR5cGU9aW50LCBkZWZhdWx0PTEwKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kZXQtc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTY0MCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbW9kZWwtbmFtZSIsIGRlZmF1bHQ9ImJ1ZmZhbG9fbCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1vZGVsLXJvb3QiLCB0eXBlPVBhdGgsIGRlZmF1bHQ9UGF0aCgifi8uaW5zaWdodGZhY2UiKSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tYWNjZXB0LW5vbmNvbW1lcmNpYWwtbW9kZWwtbGljZW5zZSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWZhaWwtZmFzdCIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICByZXR1cm4gcGFyc2VyCgoKZGVmIG1haW4oKSAtPiBpbnQ6CiAgICBhcmdzID0gYnVpbGRfcGFyc2VyKCkucGFyc2VfYXJncygpCiAgICByZXBvcnQgPSBydW5fcGlwZWxpbmUoYXJncykKICAgIHByaW50KGpzb24uZHVtcHMocmVwb3J0LCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0yKSkKICAgIHJldHVybiAwCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHJhaXNlIFN5c3RlbUV4aXQobWFpbigpKQo=', 'scripts/audit_celebdf_baseline.py': '...'}
EMBEDDED_CODE_SHA256 = "c944aaa60121560ad7e6dc0875f557e7f6077b8ff280cdafbe1314ea631d8a4e"

if IN_HOSTED_COLAB and CODE_SOURCE == "github":
    REPO_DIR = Path("/content/face-image")
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    CODE_VERSION = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
    ).strip()
elif IN_HOSTED_COLAB:
    REPO_DIR = Path("/content/face-image")
    for relative_path, encoded in EMBEDDED_FILES_B64.items():
        target = REPO_DIR / relative_path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(base64.b64decode(encoded))
    CODE_VERSION = f"embedded:{EMBEDDED_CODE_SHA256[:12]}"
else:
    REPO_DIR = Path.cwd()
    try:
        CODE_VERSION = subprocess.check_output(
            ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
        ).strip()
    except (FileNotFoundError, subprocess.SubprocessError):
        CODE_VERSION = f"local:{EMBEDDED_CODE_SHA256[:12]}"

os.chdir(REPO_DIR)
print({"repo": str(REPO_DIR), "code_source": CODE_SOURCE, "code_version": CODE_VERSION})

In [ ]:
#@title 4. Drive 연결과 runtime 작업 경로
import json
import re
import shutil

source_zip = Path(SOURCE_ZIP_PATH).expanduser()
source_transport = "configured_path"
runtime_upload_zip = Path("/content/Celeb-DF-v2.zip")
runtime_upload_parts = list(Path("/content").glob("Celeb-DF-v2.zip.part-*"))

def runtime_part_index(path):
    match = re.fullmatch(r"Celeb-DF-v2\.zip\.part-(\d+)", path.name)
    if match is None:
        raise ValueError(f"Malformed runtime upload part name: {path.name}")
    return int(match.group(1))

runtime_upload_parts.sort(key=runtime_part_index)
runtime_upload_zip_matches_expected = runtime_upload_zip.exists() and (
    bool(EXPECTED_SOURCE_ZIP_BYTES)
    and runtime_upload_zip.stat().st_size == EXPECTED_SOURCE_ZIP_BYTES
)
if IN_HOSTED_COLAB and runtime_upload_zip.exists() and not runtime_upload_zip_matches_expected:
    print({
        "ignored_runtime_upload_zip_bytes": runtime_upload_zip.stat().st_size,
        "expected_source_zip_bytes": EXPECTED_SOURCE_ZIP_BYTES,
    })

if IN_HOSTED_COLAB and ASSEMBLE_RUNTIME_UPLOAD_PARTS:
    if not runtime_upload_parts:
        raise FileNotFoundError("No /content/Celeb-DF-v2.zip.part-* uploads were found.")
    part_indices = [runtime_part_index(part) for part in runtime_upload_parts]
    if part_indices != list(range(len(part_indices))):
        raise ValueError(
            f"Runtime upload part indices must be consecutive from 0: {part_indices}"
        )
    expected_joined_bytes = sum(part.stat().st_size for part in runtime_upload_parts)
    temporary_joined_zip = runtime_upload_zip.with_suffix(".zip.joining")
    with temporary_joined_zip.open("wb") as sink:
        for part in runtime_upload_parts:
            with part.open("rb") as source:
                shutil.copyfileobj(source, sink, length=64 * 1024 * 1024)
    if temporary_joined_zip.stat().st_size != expected_joined_bytes:
        raise IOError(
            f"runtime upload join size mismatch: "
            f"{temporary_joined_zip.stat().st_size} != {expected_joined_bytes}"
        )
    temporary_joined_zip.replace(runtime_upload_zip)
    source_zip = runtime_upload_zip
    source_transport = "runtime_upload_parts"
elif IN_HOSTED_COLAB and runtime_upload_zip_matches_expected:
    source_zip = runtime_upload_zip
    source_transport = "runtime_upload"
elif IN_HOSTED_COLAB and DRIVE_SOURCE_FILE_ID.strip():
    from google.colab import auth
    from google.auth import default
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload

    auth.authenticate_user()
    credentials, _ = default()
    drive_service = build("drive", "v3", credentials=credentials, cache_discovery=False)
    metadata = drive_service.files().get(
        fileId=DRIVE_SOURCE_FILE_ID.strip(), fields="id,name,size"
    ).execute()
    expected_size = int(metadata["size"])
    source_zip = Path("/content/Celeb-DF-v2.zip")
    if not source_zip.exists() or source_zip.stat().st_size != expected_size:
        temporary_zip = source_zip.with_suffix(".zip.part")
        request = drive_service.files().get_media(fileId=DRIVE_SOURCE_FILE_ID.strip())
        with temporary_zip.open("wb") as handle:
            downloader = MediaIoBaseDownload(handle, request, chunksize=64 * 1024 * 1024)
            done = False
            while not done:
                status, done = downloader.next_chunk()
                if status:
                    print({"drive_api_download_percent": round(status.progress() * 100, 1)})
        if temporary_zip.stat().st_size != expected_size:
            raise IOError(
                f"Drive API download size mismatch: {temporary_zip.stat().st_size} != {expected_size}"
            )
        temporary_zip.replace(source_zip)
    print({"drive_api_source": metadata["name"], "bytes": expected_size})
    source_transport = "drive_api"
elif IN_HOSTED_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    source_transport = "drivefs"

DRIVE_MOUNTED = source_transport == "drivefs"

if not source_zip.exists():
    raise FileNotFoundError(f"Celeb-DF ZIP not found: {source_zip}")
if EXPECTED_SOURCE_ZIP_BYTES and source_zip.stat().st_size != EXPECTED_SOURCE_ZIP_BYTES:
    raise IOError(
        f"Celeb-DF ZIP size mismatch: {source_zip.stat().st_size} != {EXPECTED_SOURCE_ZIP_BYTES}"
    )

DATA_ROOT = Path("/content/celebdf_faceguard") if IN_HOSTED_COLAB else REPO_DIR / "outputs" / "celebdf_faceguard"
AUDIT_ROOT = Path("/content/celebdf_baseline_audit") if IN_HOSTED_COLAB else REPO_DIR / "outputs" / "celebdf_baseline_audit"
VIDEO_ROOT = DATA_ROOT / "videos"
MANIFEST = DATA_ROOT / "celeb_real_manifest.csv"
INVENTORY_JSON = DATA_ROOT / "celeb_real_inventory.json"
SANITIZED_ROOT = AUDIT_ROOT / "sanitized"
for path in (DATA_ROOT, AUDIT_ROOT, SANITIZED_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print({
    "source_zip_gb": round(source_zip.stat().st_size / 1e9, 3),
    "source_transport": source_transport,
    "drive_mounted": DRIVE_MOUNTED,
    "runtime_free_gb": round(shutil.disk_usage(AUDIT_ROOT).free / 1e9, 2),
    "audit_root": str(AUDIT_ROOT),
    "sanitized_drive_dir": DRIVE_RESULT_DIR if PERSIST_SANITIZED_RESULTS_TO_DRIVE else None,
})

In [ ]:
#@title 5. ZIP inventory와 Celeb-real 590개 추출
subprocess.run([
    sys.executable, "scripts/celebdf_faceguard.py", "inventory", str(source_zip),
    "--manifest", str(MANIFEST), "--summary", str(INVENTORY_JSON),
], check=True)
inventory = json.loads(INVENTORY_JSON.read_text(encoding="utf-8"))
assert inventory["video_count"] == 590, inventory
assert inventory["subject_count"] == 59, inventory
assert inventory["eligible_subjects_ge_8_videos"] == 56, inventory

subprocess.run([
    sys.executable, "scripts/celebdf_faceguard.py", "extract", str(source_zip),
    "--manifest", str(MANIFEST), "--output", str(VIDEO_ROOT), "--mode", "full",
], check=True)
extracted = sorted((VIDEO_ROOT / "Celeb-real").glob("*.mp4"))
if len(extracted) != 590:
    raise RuntimeError(f"Expected 590 videos, found {len(extracted)}")
print({"videos": len(extracted), "eligible_subjects": 56})

In [ ]:
#@title 6. GPU/ONNX Runtime 확인
import subprocess
import onnxruntime as ort

providers = ort.get_available_providers()
print({"onnxruntime": ort.__version__, "providers": providers})
if IN_HOSTED_COLAB and "CUDAExecutionProvider" not in providers:
    raise RuntimeError("CUDAExecutionProvider is unavailable. Restart a GPU runtime without rerunning cell 2.")
print(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True,
))

In [ ]:
#@title 7. frames 1/5/10 전체 ArcFace 추론
EMBEDDING_RUNS = {}
RUN_REPORTS = {}
REJECT_FILES = {}

for frames in FRAME_VALUES:
    run_root = AUDIT_ROOT / f"frames_{frames}"
    run_root.mkdir(parents=True, exist_ok=True)
    embeddings = run_root / "video_embeddings.npz"
    rejects = run_root / "rejects.csv"
    run_report = run_root / "run.json"
    minimum_valid_frames = min(3, frames)
    command = [
        sys.executable, "scripts/run_celebdf_arcface.py",
        "--manifest", str(MANIFEST),
        "--video-root", str(VIDEO_ROOT),
        "--output", str(embeddings),
        "--rejects", str(rejects),
        "--run-report", str(run_report),
        "--frames-per-video", str(frames),
        "--minimum-valid-frames", str(minimum_valid_frames),
        "--checkpoint-every", "25",
        "--progress-every", "25",
        "--model-name", "buffalo_l",
        "--accept-noncommercial-model-license",
    ]
    if RUN_SMOKE_BEFORE_FULL and frames == FRAME_VALUES[0] and not embeddings.exists():
        subprocess.run(
            command + ["--mode", "smoke", "--smoke-subjects", "2", "--smoke-videos-per-subject", "1"],
            check=True,
        )
    subprocess.run(command + ["--mode", "full"], check=True)
    EMBEDDING_RUNS[frames] = embeddings
    RUN_REPORTS[frames] = run_report
    REJECT_FILES[frames] = rejects

print({
    "completed_frame_runs": sorted(EMBEDDING_RUNS),
    "embedding_files_stay_in_runtime": True,
})

In [ ]:
#@title 8. 다중 seed·reference 감사와 누수 검사
audit_command = [
    sys.executable, "scripts/audit_celebdf_baseline.py",
    "--output-dir", str(SANITIZED_ROOT),
    "--seeds", ",".join(str(value) for value in SEED_VALUES),
    "--reference-counts", ",".join(str(value) for value in REFERENCE_VALUES),
    "--bootstrap-repeats", str(BOOTSTRAP_REPEATS),
    "--max-reference-count", "5",
]
for frames in FRAME_VALUES:
    audit_command.extend(["--embedding-run", f"{frames}={EMBEDDING_RUNS[frames]}"])
    audit_command.extend(["--run-report", f"{frames}={RUN_REPORTS[frames]}"])
    if REJECT_FILES[frames].exists():
        audit_command.extend(["--rejects", f"{frames}={REJECT_FILES[frames]}"])
subprocess.run(audit_command, check=True)

AUDIT_JSON = SANITIZED_ROOT / "celebdf_baseline_audit.json"
METRICS_CSV = SANITIZED_ROOT / "celebdf_baseline_audit_metrics.csv"
SUMMARY_CSV = SANITIZED_ROOT / "celebdf_baseline_audit_summary.csv"
audit_report = json.loads(AUDIT_JSON.read_text(encoding="utf-8"))
assert all(item["validation_test_identity_overlap"] == 0 for item in audit_report["leakage_checks"])
assert all(item["registration_query_video_overlap"] == 0 for item in audit_report["leakage_checks"])
print(json.dumps(audit_report["decisions"], ensure_ascii=False, indent=2))

In [ ]:
#@title 9. seed 변동과 운영점 결과 확인
import pandas as pd

metrics = pd.read_csv(METRICS_CSV)
summary = pd.read_csv(SUMMARY_CSV)
display(summary[[
    "frames_per_video", "reference_count",
    "test_roc_auc_mean", "test_roc_auc_min",
    "test_eer_mean", "test_eer_max",
    "far_0.001_test_tar_mean", "far_0.001_test_far_mean",
    "far_0.001_threshold_mean", "far_0.001_threshold_std",
]])
display(pd.DataFrame([
    {
        "frames_per_video": item["frames_per_video"],
        **item["quality"],
        "reject_reason_counts": item["reject_reason_counts"],
    }
    for item in audit_report["input_runs"]
]))

In [ ]:
#@title 10. 감사 그래프 생성
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.lineplot(
    data=summary,
    x="frames_per_video",
    y="far_0.001_test_tar_mean",
    hue="reference_count",
    marker="o",
    palette="viridis",
    ax=axes[0],
)
axes[0].set(title="Mean TAR at validation-selected FAR=0.001", ylabel="Test TAR")
sns.lineplot(
    data=summary,
    x="frames_per_video",
    y="far_0.001_test_far_mean",
    hue="reference_count",
    marker="o",
    palette="viridis",
    ax=axes[1],
    legend=False,
)
axes[1].axhline(0.001, color="red", linestyle="--", linewidth=1, label="target FAR")
axes[1].set(title="Observed mean Test FAR", ylabel="Test FAR")
axes[1].legend()
fig.tight_layout()
FIGURE_PNG = SANITIZED_ROOT / "celebdf_baseline_audit.png"
fig.savefig(FIGURE_PNG, dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
#@title 11. 비식별 결과 ZIP 생성과 Drive 보존
import zipfile

RUNTIME_CONFIG = SANITIZED_ROOT / "audit_runtime_config.json"
RUNTIME_CONFIG.write_text(json.dumps({
    "code_version": CODE_VERSION,
    "frames_per_video": FRAME_VALUES,
    "reference_counts": REFERENCE_VALUES,
    "max_reference_count": 5,
    "seeds": SEED_VALUES,
    "bootstrap_repeats": BOOTSTRAP_REPEATS,
    "raw_data_in_bundle": False,
    "embeddings_in_bundle": False,
}, ensure_ascii=False, indent=2), encoding="utf-8")

RESULT_BUNDLE = SANITIZED_ROOT / "celebdf_baseline_audit_results.zip"
with zipfile.ZipFile(RESULT_BUNDLE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in (AUDIT_JSON, METRICS_CSV, SUMMARY_CSV, FIGURE_PNG, RUNTIME_CONFIG):
        archive.write(path, arcname=path.name)

saved_to = None
if PERSIST_SANITIZED_RESULTS_TO_DRIVE and DRIVE_MOUNTED:
    drive_result_dir = Path(DRIVE_RESULT_DIR)
    drive_result_dir.mkdir(parents=True, exist_ok=True)
    saved_to = shutil.copy2(RESULT_BUNDLE, drive_result_dir / RESULT_BUNDLE.name)
elif IN_HOSTED_COLAB:
    from google.colab import files
    files.download(str(RESULT_BUNDLE))
print({
    "result_bundle": str(RESULT_BUNDLE),
    "size_mb": round(RESULT_BUNDLE.stat().st_size / 1e6, 2),
    "saved_to_drive": str(saved_to) if saved_to else None,
})

## 해석 제한

- 완벽한 ROC-AUC가 반복돼도 Celeb-real이 쉬운 내부 benchmark일 가능성을 먼저 고려한다.
- validation에서 고른 threshold의 test FAR 변동을 ROC-AUC보다 우선 확인한다.
- 결과 ZIP에는 집계값과 hash만 있으며 NPZ embedding은 포함하지 않는다.
- AI-Hub 승인 후 한국인 얼굴 데이터에는 동일 프로토콜을 별도 Issue로 적용한다.